# simplicio-video · render no Colab (GPU)

Renderiza um **bundle** gerado por `simplicio-video render <contrato.yaml> --backend colab` e devolve o MP4.
Imprime o tempo de cada etapa (setup, frames, encode, master) e a GPU usada, para comparar com o render local
(o `bundle.json` traz os hashes e tempos do render local de referência quando existem).

1. **Ambiente de execução → Alterar tipo → GPU** (T4 do plano gratuito basta). Nada aqui cria conta ou gasta créditos.
2. Rode as células na ordem. Na célula 3, envie o `.colab-bundle.zip` (ou aponte `BUNDLE` para um arquivo no Drive).
3. A última célula baixa o MP4. Suba no Drive em `drive.caminho` do contrato (pasta `simplicio-video/clientes/...`).

Determinismo: com o mesmo Chrome/HyperFrames e encode por CPU, os frames saem idênticos aos locais; com NVENC (`--gpu`)
o vídeo é recodificado por outro encoder, então os bytes mudam (o áudio masterizado deve bater).

In [ ]:
# 1) GPU + relógio
import json, os, re, subprocess, time
T = {}
T0 = time.time()
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "nenhuma (runtime CPU)")

In [ ]:
# 2) Setup: Node 22 (mesma major do box), ffmpeg, libs do Chrome headless, HyperFrames fixado
t = time.time()
NODE = "v22.23.3"
sh = lambda c: subprocess.run(c, shell=True, check=True, capture_output=True, text=True).stdout
if not sh("node --version || true").startswith("v22"):
    sh(f"curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1")
sh("apt-get -qq update && apt-get -qq install -y ffmpeg libnss3 libatk-bridge2.0-0 libgbm1 libxkbcommon0 libasound2 "
   "libxcomposite1 libxdamage1 libxrandr2 libpango-1.0-0 libcairo2 > /dev/null")
print("node", sh("node --version").strip(), "|", sh("ffmpeg -version").splitlines()[0])
nvenc = "h264_nvenc" in sh("ffmpeg -hide_banner -encoders")
print("NVENC no ffmpeg:", nvenc)
print(sh("npx --yes hyperframes@0.8.97 --version").strip())
T["setup_s"] = round(time.time() - t, 1)
print("setup:", T["setup_s"], "s")

In [ ]:
# 3) Bundle: upload (padrão) ou caminho no Drive
BUNDLE = ""   # ex.: "/content/drive/MyDrive/simplicio-video/bundles/wjr-video-01-cli.colab-bundle.zip"
if not BUNDLE:
    from google.colab import files
    up = files.upload()
    BUNDLE = next(iter(up))
sh(f"rm -rf /content/bundle && mkdir -p /content/bundle && cd /content/bundle && unzip -q '{os.path.abspath(BUNDLE)}'")
meta = json.load(open("/content/bundle/bundle.json"))
print(json.dumps({k: meta.get(k) for k in ("contract", "inputs_sha256", "quality", "output_name")}, indent=1, ensure_ascii=False))

In [ ]:
# 4) Render (frames + encode) + master -14 LUFS; GPU encode quando houver NVENC
t = time.time()
p = subprocess.run("cd /content/bundle && bash render_bundle.sh", shell=True, capture_output=True, text=True)
print(p.stdout[-2500:], p.stderr[-2500:])
assert p.returncode == 0, "render falhou"
tim = json.loads(re.search(r"TIMINGS (\{.*\})", p.stdout).group(1))
T.update({"frames_capture_s": tim["capture_s"], "encode_s": tim["encode_s"], "render_total_s": tim["render_s"],
          "master_s": tim["master_s"], "gpu_encode": tim["gpu_encode"]})
T["total_s"] = round(time.time() - T0, 1)

In [ ]:
# 5) Tempos por etapa + comparação com o render local de referência
out = "/content/bundle/out/final.mp4"
dec = lambda a: subprocess.run(f"ffmpeg -v error -i {out} {a} -", shell=True, capture_output=True).stdout
import hashlib
frames = [l.rsplit(",", 1)[-1].strip() for l in dec("-map 0:v:0 -f framemd5").decode().splitlines() if l and not l.startswith("#")]
res = {"gpu": gpu or "cpu", **T, "frames": len(frames),
       "video_frames_sha256": hashlib.sha256("\n".join(frames).encode()).hexdigest(),
       "audio_pcm_sha256": hashlib.sha256(dec("-map 0:a:0 -f s16le -ac 2 -ar 48000")).hexdigest()}
ref = meta.get("reference_local")
if ref:
    res["local_timings_s"] = ref.get("timings_s")
    res["frames_iguais_ao_local"] = res["video_frames_sha256"] == ref["video_frames_sha256"]
    res["audio_igual_ao_local"] = res["audio_pcm_sha256"] == ref["audio_pcm_sha256"]
print(json.dumps(res, indent=1, ensure_ascii=False))

In [ ]:
# 6) Baixar o MP4 (nome do contrato). Upload no Drive: pasta drive.caminho do contrato.
from google.colab import files
dst = "/content/" + meta["output_name"]
sh(f"cp {out} '{dst}'")
print("Drive:", meta["drive"]["caminho"], "->", meta["drive"]["arquivo"])
files.download(dst)